In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [5]:
loader = PyPDFLoader("data_science_syllabus.pdf")
docs = loader.load()

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

In [11]:
len(splitted_docs)

11

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding=embeddings
)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

#### Agent = tools, llm, prompt

In [15]:
@tool
def retrieval_tool(query:str):
    """
    This tool can help you to retrieve the data from the PDF documents and these documents
    have detaisl about medical reports.
    """
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    for doc in docs:
        context += doc.page_content + "\n"

    return context


In [18]:
llm = ChatOpenAI(model="gpt-3.5-turbo")


In [19]:
System_prompt = """
  You are a helpful assistant for answering the questions related to the data science syllabus.
  You have access to a tool called retrieval_tool which can help you to retrieve the data from the PDF documents and these documents"""

In [21]:
agent = create_agent(
    model = llm,
    tools = [retrieval_tool],
    system_prompt = System_prompt
)

In [22]:
query = "What are the topics covered in the data science syllabus?"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [23]:
result = response["messages"][-1].content

NameError: name 'response' is not defined

In [ ]:
print(result)